# s11. Controllability checks on the four matched ensembles

Two checks on the same 400 realisations as Table 1 of the manuscript, four topologies with one hundred realisations each, N = 100 elements and exactly 300 links, seeds 42 to 141. Nothing here draws a figure, and no number in Table 1 changes.

**C1. Full-control count.** The driver nodes, the fewest independent input signals that let every element be driven to any state. It is computed two ways.
- Maximum matching (Liu, Slotine & Barabási, 2011), with every undirected link taken in both directions: N_D = max(N − |M*|, 1), the form given as Eq. 17 of Liu & Barabási (2016).
- Exact controllability (Yuan et al., 2013), which is exact for undirected links of equal weight: N_D is the largest multiplicity of any eigenvalue of the adjacency matrix.

**C2. Influence check of the one assumption.** The manuscript reads control entropy as a specification cost on one assumption, namely that an element's influence on the whole is proportional to its coupling share. Average controllability (Gu et al., 2015) measures that influence in control-theoretic terms. It is the trace of the infinite-horizon controllability Gramian when the element alone receives input. Gu et al. stabilised their weighted brain networks by dividing by the mean edge weight, which leaves an unweighted graph unchanged and unstable, so here A is scaled by one plus its largest singular value, and Check 6 shows how the result moves with that constant. C2 is the Pearson correlation between share and average controllability across the elements of each realisation. It tests whether influence rises with share in a straight line, not whether the two are strictly proportional.

The first code cell is the core cell used by every notebook of the study, byte-identical. The second holds the generators exactly as published. The third reproduces Table 1 before any new number is computed.

In [1]:
# ============================================================================
# CORE  -  Equations 1 to 13 of the PNAS manuscript (Methods numbering).
# Byte-identical cell in every notebook, verified by hash (roadmap step 6.4).
# ============================================================================
# Eq 1   p_i = c_i / sum_j c_j                       coupling share
# Eq 2   H = - sum_i p_i log2 p_i                     control entropy (avg information)
# Eq 3   p_i = C_i / sum_j C_j,  A C = lambda C       eigenvector weighting
# Eq 4   J = H / log2 N                               evenness
# Eq 5   D_eff = 2^H                                  effective count of DOF
# Eq 6   D_eff / N = 2^(-dH),  dH = log2 N - H        retained fraction / compression
# Eq 7   H <= log2 N                                  the bound (Gibbs / Jensen)
# Eq 8   dH/deps = log2(p_j / p_i)                    transfer gradient, concentration lowers H
# Eq 9   H = H2(phi) + phi log2 h + (1-phi) log2(N-h) leader-follower closed form
# Eq 10  kappa = <k^2> / <k>                          degree heterogeneity
# Eq 11  r                                            degree assortativity
# Eq 12  dh = log2(sigma_A / sigma_B)                 outcome entropy difference
# Eq 13 (Hebbian) lives in s4, since it needs the parent model's activations.
#
# I(p) = log2(1/p) is the summand of Eq 2 (Introduction Eq 2).
# Compression dH = log2 N - H is the bits saved (Introduction Eq 5), used by Eq 6.
# ============================================================================
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

CORE_VERSION = "v5 / 24 July 2026 (Methods Eq 1-13)"

def share(c):
    """Eq 1. Normalise any non-negative vector into coupling shares."""
    c = np.asarray(c, dtype=float).ravel()
    if np.any(c < 0):
        raise ValueError("coupling values must be non-negative")
    t = c.sum()
    if t <= 0:
        raise ValueError("total coupling is zero")
    return c / t

def information(p):
    """Information of a share, in bits, the summand of Eq 2. Small share -> large information."""
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.inf)
    np.log2(1.0 / p, out=out, where=(p > 0))
    return out

def shannon_entropy(c):
    """Eq 2. H as the average of the information over the distribution. Input may be raw."""
    p = share(c)
    p = p[p > 0]
    return float((p * information(p)).sum())

def max_entropy(n):
    """Eq 7 (right side). The ceiling log2 N, attained only by the uniform distribution."""
    if n < 1:
        raise ValueError("n must be at least 1")
    return float(np.log2(n))

def evenness(h, n):
    """Eq 4. J = H / log2 N. Compare systems of different size with this."""
    m = max_entropy(n)
    return 1.0 if m == 0 else float(h / m)

def effective_dof(h):
    """Eq 5. D_eff = 2^H, a number of equally coupled degrees of freedom."""
    return float(2.0 ** h)

def compression(H, N):
    """dH = log2 N - H (Introduction Eq 5). Bits saved against even coupling.
    Regular gives 0 exactly; the scale-free ensemble gives 0.326 at N = 100.
    """
    return float(np.log2(N) - H)

def retained_fraction(H, N):
    """Eq 6. 2^(-dH) = D_eff / N, the fraction of degrees of freedom still to be
    specified. Equals 2^H / N. Scale-free gives 0.798, the 79.8 out of 100.
    """
    return float(2.0 ** (-(np.log2(N) - H)))

def coupling_shares(G, weight="degree"):
    """Eq 1 with c = degree, or Eq 3 with c = eigenvector centrality."""
    nodes = list(G.nodes())
    if weight == "degree":
        v = np.array([G.degree(n) for n in nodes], dtype=float)
    elif weight == "eigenvector":
        c = nx.eigenvector_centrality_numpy(G)
        v = np.abs(np.array([c[n] for n in nodes], dtype=float))
    else:
        raise ValueError("weight must be 'degree' or 'eigenvector'")
    return share(v)

def control_entropy(G, weight="degree"):
    """Eq 2 applied to Eq 1 or Eq 3."""
    return shannon_entropy(coupling_shares(G, weight=weight))

def strength_shares(W):
    """Eq 1 for a weighted structure: c_i = node strength = sum_j W_ij."""
    W = np.asarray(W, dtype=float)
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError("W must be square")
    return share(W.sum(axis=1))

def transfer_gradient(p, i, j):
    """Eq 8. dH/deps = log2(p_j / p_i) when eps moves coupling FROM j TO i.
    Negative whenever p_i > p_j, so any transfer toward an already-strong
    element strictly lowers H. i is the element that receives coupling.
    """
    p = np.asarray(p, dtype=float)
    return float(np.log2(p[j] / p[i]))

def _binary_entropy(phi):
    """H2(phi) = -phi log2 phi - (1-phi) log2(1-phi), with the 0 log 0 = 0 rule."""
    phi = float(phi)
    if phi <= 0.0 or phi >= 1.0:
        return 0.0
    return float(-phi * np.log2(phi) - (1.0 - phi) * np.log2(1.0 - phi))

def two_level_entropy(N, h, phi):
    """Eq 9. H = H2(phi) + phi log2 h + (1-phi) log2(N-h).
    h leaders share phi equally and N-h followers share (1-phi) equally.
    Returns log2 N exactly when phi = h/N, the built-in unit test.
    """
    if not (1 <= h < N):
        raise ValueError("need 1 <= h < N")
    if not (0.0 <= phi <= 1.0):
        raise ValueError("phi must be in [0, 1]")
    lead = phi * np.log2(h) if h > 0 else 0.0
    return float(_binary_entropy(phi) + lead + (1.0 - phi) * np.log2(N - h))

def degree_heterogeneity(G):
    """Eq 10. kappa = <k^2> / <k>, the parent paper's normalisation.

    NOTE ON NORMALISATION. The handwritten note of 23 July writes the ratio as
    <k^2>/<k>^2. That differs from this by a factor <k>, and the parent paper's
    published values (6.84, 6.09, 9.67) are on THIS definition. Table 1 of the
    manuscript uses this one so the two papers stay comparable. Convert with
    kappa_alt = kappa / <k> if a source uses the other.
    """
    k = np.array([d for _, d in G.degree()], dtype=float)
    if k.mean() <= 0:
        return float("nan")
    return float((k ** 2).mean() / k.mean())

def degree_assortativity(G):
    """Eq 11. Pearson correlation of degrees at the two ends of an edge.

    Undefined for a regular graph, where the degrees carry no variance. Returns
    nan there rather than a fabricated zero.
    """
    k = np.array([d for _, d in G.degree()], dtype=float)
    if k.std() == 0 or G.number_of_edges() == 0:
        return float("nan")
    try:
        r = nx.degree_assortativity_coefficient(G)
    except (ValueError, ZeroDivisionError):
        return float("nan")
    return float(r) if np.isfinite(r) else float("nan")

def gini(x):
    """Gini coefficient, matching the parent paper's definition."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2.0 * (idx * x).sum()) / (n * x.sum()) - (n + 1.0) / n)

GAUSS_CONST = 0.5 * np.log2(2.0 * np.pi * np.e)

def outcome_entropy_gaussian(sigma):
    """h = 0.5 log2(2 pi e sigma^2). A DIFFERENTIAL entropy, not Eq 2.

    It shares the unit and the reading of predictability with control entropy
    and is a different quantity on a different object. Only differences on a
    common scale are interpreted, which is why Eq 12 exists.
    """
    s = np.asarray(sigma, dtype=float)
    if np.any(s <= 0):
        raise ValueError("sigma must be positive")
    return GAUSS_CONST + np.log2(s)

def delta_outcome_entropy(sigma_a, sigma_b):
    """Eq 12. The additive constant and the unit both cancel."""
    return float(np.log2(float(sigma_a) / float(sigma_b)))

def predictability_ratio(dh):
    """R = 2^dh, the predictability ratio that follows Eq 12."""
    return float(2.0 ** float(dh))

print("core loaded:", CORE_VERSION)

core loaded: v5 / 24 July 2026 (Methods Eq 1-13)


In [2]:
import sys, platform, csv
import numpy as np
import networkx as nx
import scipy
from scipy import stats
from networkx.algorithms import bipartite
print('Python', platform.python_version(), '| NumPy', np.__version__,
      '| NetworkX', nx.__version__, '| SciPy', scipy.__version__)

# The four matched ensembles, generators as published (Supplementary Note 1).
N, K_MEAN, N_REAL, SEED0 = 100, 6, 100, 42
M_LINKS = N * K_MEAN // 2                # 300 links in every realisation
BA_M, BA_N0 = 3, 12                      # 36 + 3 x 88 = 300


def make(kind, seed):
    if kind == 'regular':     return nx.watts_strogatz_graph(N, K_MEAN, 0.0, seed=seed)
    if kind == 'small-world': return nx.watts_strogatz_graph(N, K_MEAN, 0.1, seed=seed)
    if kind == 'random':      return nx.gnm_random_graph(N, M_LINKS, seed=seed)
    if kind == 'scalefree':
        g0 = nx.watts_strogatz_graph(BA_N0, K_MEAN, 0.0, seed=seed)
        return nx.barabasi_albert_graph(N, BA_M, seed=seed, initial_graph=g0)
    raise ValueError(kind)


def giant(G):
    return G if nx.is_connected(G) else G.subgraph(
        max(nx.connected_components(G), key=len)).copy()


KINDS = ['regular', 'small-world', 'random', 'scalefree']
LABEL = {'regular': 'Regular lattice', 'small-world': 'Small-world',
         'random': 'Random', 'scalefree': 'Scale-free'}
SEEDS = range(SEED0, SEED0 + N_REAL)
GRAPHS = {k: [make(k, s) for s in SEEDS] for k in KINDS}
print('realisations:', {k: len(v) for k, v in GRAPHS.items()})

Python 3.11.15 | NumPy 2.4.4 | NetworkX 3.6.1 | SciPy 1.17.1
realisations: {'regular': 100, 'small-world': 100, 'random': 100, 'scalefree': 100}


## Reproduction of Table 1

Every value printed in Table 1 must come back to the printed precision before C1 and C2 are computed, which shows these are the same 400 networks.

In [3]:
TABLE1 = {   # H, J, 2^H, compression, kappa, as printed in Table 1
    'regular':     (6.6439, 1.0000, 100.0, 0.000,  6.00),
    'small-world': (6.5832, 0.9909,  95.9, 0.061,  6.09),
    'random':      (6.4820, 0.9756,  89.4, 0.162,  6.90),
    'scalefree':   (6.3176, 0.9509,  79.8, 0.326, 10.10),
}
H_EIG = {}
for kind in KINDS:
    H = np.array([shannon_entropy(coupling_shares(giant(G), 'eigenvector'))
                  for G in GRAPHS[kind]])
    kap = np.array([degree_heterogeneity(G) for G in GRAPHS[kind]])
    links = sorted({G.number_of_edges() for G in GRAPHS[kind]})
    Hm = H.mean()
    got = (round(Hm, 4), round(evenness(Hm, N), 4), round(effective_dof(Hm), 1),
           round(max(compression(Hm, N), 0.0), 3), round(kap.mean(), 2))
    H_EIG[kind] = H
    print(f"{LABEL[kind]:16s} H {got[0]:.4f}  J {got[1]:.4f}  2^H {got[2]:5.1f}  "
          f"dH {got[3]:.3f}  kappa {got[4]:5.2f}  links {links}  "
          f"{'matches Table 1' if got == TABLE1[kind] else 'MISMATCH'}")
    assert got == TABLE1[kind], (kind, got, TABLE1[kind])
print('Table 1 reproduced to the printed precision.')

Regular lattice  H 6.6439  J 1.0000  2^H 100.0  dH 0.000  kappa  6.00  links [300]  matches Table 1
Small-world      H 6.5832  J 0.9909  2^H  95.9  dH 0.061  kappa  6.09  links [300]  matches Table 1
Random           H 6.4820  J 0.9756  2^H  89.4  dH 0.162  kappa  6.90  links [300]  matches Table 1
Scale-free       H 6.3176  J 0.9509  2^H  79.8  dH 0.326  kappa 10.10  links [300]  matches Table 1
Table 1 reproduced to the printed precision.


## C1. Full-control count

Both counts are taken on the full generated graph, as κ and the assortativity are. The unit tests use the four simple graphs whose exact counts the review tabulates, chain 1, ring 2, star N − 2 and complete graph N − 1 (Liu & Barabási, 2016).

In [4]:
def driver_nodes_matching(G):
    """Liu et al. (2011). N_D = max(N - |M*|, 1), with M* a maximum matching
    of the directed network. Each undirected link is taken in both directions."""
    out_side = [('out', n) for n in G.nodes()]
    in_side = [('in', n) for n in G.nodes()]
    B = nx.Graph()
    B.add_nodes_from(out_side, bipartite=0)
    B.add_nodes_from(in_side, bipartite=1)
    for u, v in G.edges():
        B.add_edge(('out', u), ('in', v))
        B.add_edge(('out', v), ('in', u))
    M = bipartite.hopcroft_karp_matching(B, top_nodes=out_side)
    matched = sum(1 for key in M if key[0] == 'out')
    return max(G.number_of_nodes() - matched, 1)


def driver_nodes_exact(G, tol=1e-6):
    """Yuan et al. (2013). For a symmetric adjacency matrix the exact count is
    the largest multiplicity of any eigenvalue, max over lambda of
    N - rank(lambda I - A). Returns the count and that eigenvalue."""
    A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
    ev = np.sort(np.linalg.eigvalsh(A))
    groups = np.split(ev, np.where(np.diff(ev) > tol)[0] + 1)
    top = max(groups, key=len)
    lam = float(top.mean())
    by_rank = A.shape[0] - np.linalg.matrix_rank(lam * np.eye(A.shape[0]) - A, tol=tol)
    assert by_rank == len(top), (by_rank, len(top))   # multiplicity confirmed by rank
    return int(len(top)), lam


n = 12
tests = {'chain': (nx.path_graph(n), 1), 'ring': (nx.cycle_graph(n), 2),
         'star': (nx.star_graph(n - 1), n - 2), 'complete': (nx.complete_graph(n), n - 1)}
for name, (G, want) in tests.items():
    got, lam = driver_nodes_exact(G)
    print(f"{name:9s} N = {n}: exact {got:2d} (review {want:2d}), "
          f"matching {driver_nodes_matching(G):2d}")
    assert got == want

chain     N = 12: exact  1 (review  1), matching  1
ring      N = 12: exact  2 (review  2), matching  1
star      N = 12: exact 10 (review 10), matching 10
complete  N = 12: exact 11 (review 11), matching  1


## C2. Average controllability

For a symmetric A the infinite-horizon Gramian with input at element i alone has trace [(I − Â²)⁻¹]ᵢᵢ, where Â = A / (1 + σ_max). The cell checks that closed form against the Gramian summed term by term. Shares and average controllability are both taken on the largest component, where the eigenvector shares of Table 1 are defined.

In [5]:
def average_controllability(G, c=1.0):
    """Average controllability as defined by Gu et al. (2015): the trace of the
    infinite-horizon controllability Gramian with B = e_i, which is
    [(I - A^2)^-1]_ii for a symmetric A. A is first scaled by c + its largest
    singular value so that x(t+1) = A x(t) + B u(t) is stable and the sum
    converges. Returns AC_i for every element, in G.nodes() order."""
    nodes = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes)
    As = A / (c + np.linalg.norm(A, 2))
    return np.diag(np.linalg.inv(np.eye(len(nodes)) - As @ As))


def share_influence_r(G, weight='eigenvector', c=1.0, method='pearson'):
    """C2. Correlation of coupling share with average controllability across
    the elements of one structure. Undefined when every share is equal."""
    p = coupling_shares(G, weight)
    if np.ptp(p) < 1e-12:
        return float('nan')
    ac = average_controllability(G, c)
    if method == 'spearman':
        # round first, so that shares equal up to floating noise tie the same way every run
        return float(stats.spearmanr(np.round(p, 12), np.round(ac, 12)).correlation)
    return float(np.corrcoef(p, ac)[0, 1])


# closed form against the Gramian summed term by term, one scale-free realisation
G = GRAPHS['scalefree'][0]
A = nx.to_numpy_array(G, nodelist=list(G.nodes()))
As = A / (1.0 + np.linalg.norm(A, 2))
series, term = np.zeros(len(A)), np.eye(len(A))
for _ in range(4000):
    series += np.diag(term @ term.T)
    term = As @ term
gap = np.max(np.abs(series - average_controllability(G)))
print(f"closed form against the summed Gramian: largest difference {gap:.1e}")
assert gap < 1e-10

closed form against the summed Gramian: largest difference 2.4e-15


## All 400 realisations

In [6]:
rows = []
for kind in KINDS:
    for seed, Gf in zip(SEEDS, GRAPHS[kind]):
        G = giant(Gf)
        nd_m = driver_nodes_matching(Gf)
        nd_e, _ = driver_nodes_exact(Gf)
        rows.append({
            'topology': kind, 'seed': seed,
            'elements_outside_largest_component': Gf.number_of_nodes() - G.number_of_nodes(),
            'H_eigenvector': shannon_entropy(coupling_shares(G, 'eigenvector')),
            'driver_nodes_matching': nd_m,
            'driver_nodes_exact': nd_e,
            'r_share_avgctrl': share_influence_r(G),
            'r_degreeshare_avgctrl': share_influence_r(G, weight='degree'),
            'spearman_share_avgctrl': share_influence_r(G, method='spearman'),
            'r_share_avgctrl_c0.5': share_influence_r(G, c=0.5),
            'r_share_avgctrl_c2': share_influence_r(G, c=2.0),
        })
with open('s11_controllability_checks.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    for r in rows:
        w.writerow({k: (f'{v:.6f}' if isinstance(v, float) else v) for k, v in r.items()})
print(len(rows), 'rows written to s11_controllability_checks.csv')

400 rows written to s11_controllability_checks.csv


In [7]:
def col(kind, key):
    return np.array([r[key] for r in rows if r['topology'] == kind], dtype=float)


rng = np.random.default_rng(0)
def boot_ci(v, n=10000):
    v = v[np.isfinite(v)]
    means = np.array([rng.choice(v, size=v.size, replace=True).mean() for _ in range(n)])
    return np.percentile(means, [2.5, 97.5])


SUMMARY = {}
print('C1  driver nodes for full control, out of 100 elements')
print(f"{'':16s}{'matching':>24s}{'exact':>24s}{'one driver, matching / exact':>32s}")
for kind in KINDS:
    m, e = col(kind, 'driver_nodes_matching'), col(kind, 'driver_nodes_exact')
    SUMMARY[kind] = {'m': m, 'e': e}
    print(f"{LABEL[kind]:16s}{m.mean():8.2f} ± {m.std(ddof=1):4.2f} ({m.min():.0f} to {m.max():.0f})"
          f"{e.mean():8.2f} ± {e.std(ddof=1):4.2f} ({e.min():.0f} to {e.max():.0f})"
          f"{np.mean(m == 1) * 100:16.0f} % / {np.mean(e == 1) * 100:.0f} %")

print()
print('C2  Pearson r, coupling share against average controllability, within each realisation')
for kind in KINDS:
    r = col(kind, 'r_share_avgctrl')
    SUMMARY[kind]['r'] = r
    if np.all(np.isnan(r)):
        print(f"{LABEL[kind]:16s} undefined, every element holds the same share and the same influence")
        continue
    lo, hi = boot_ci(r)
    print(f"{LABEL[kind]:16s} mean {r.mean():.3f} (95% CI {lo:.3f} to {hi:.3f}), "
          f"sd {r.std(ddof=1):.3f}, range {r.min():.3f} to {r.max():.3f}, "
          f"positive in {np.mean(r > 0) * 100:.0f} % of realisations")

def cohen_d(a, b):
    sp = np.sqrt(((a.size - 1) * a.var(ddof=1) + (b.size - 1) * b.var(ddof=1)) / (a.size + b.size - 2))
    return (a.mean() - b.mean()) / sp

print()
print('How strongly each count separates scale-free from random, Cohen d')
print(f"  control entropy (Table 1)      d = {cohen_d(H_EIG['random'], H_EIG['scalefree']):.1f}")
print(f"  driver nodes, matching         d = {cohen_d(SUMMARY['scalefree']['m'], SUMMARY['random']['m']):.1f}")
print(f"  driver nodes, exact            d = {cohen_d(SUMMARY['scalefree']['e'], SUMMARY['random']['e']):.1f}")

C1  driver nodes for full control, out of 100 elements
                                matching                   exact    one driver, matching / exact
Regular lattice     1.00 ± 0.00 (1 to 1)    3.00 ± 0.00 (3 to 3)             100 % / 0 %
Small-world         1.00 ± 0.00 (1 to 1)    1.00 ± 0.00 (1 to 1)             100 % / 100 %
Random              1.01 ± 0.10 (1 to 2)    1.01 ± 0.10 (1 to 2)              99 % / 99 %
Scale-free          2.17 ± 2.05 (1 to 10)    2.24 ± 2.09 (1 to 10)              60 % / 58 %

C2  Pearson r, coupling share against average controllability, within each realisation
Regular lattice  undefined, every element holds the same share and the same influence
Small-world      mean 0.757 (95% CI 0.744 to 0.769), sd 0.065, range 0.547 to 0.883, positive in 100 % of realisations
Random           mean 0.964 (95% CI 0.962 to 0.965), sd 0.006, range 0.941 to 0.976, positive in 100 % of realisations
Scale-free       mean 0.969 (95% CI 0.968 to 0.970), sd 0.006, range 0.948

## Checks

Each check leaves the reported numbers in place or shows by how much they move.

In [8]:
# 1. The exact count does not depend on the tolerance used to group eigenvalues.
same = all(driver_nodes_exact(G, 1e-6)[0] == driver_nodes_exact(G, 1e-9)[0]
           for kind in KINDS for G in GRAPHS[kind])
print('1. exact count unchanged between tolerances 1e-6 and 1e-9:', same)

# 2. The matching count, recomputed with SciPy's independent bipartite matching.
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching
def nd_scipy(G):
    nodes = list(G.nodes()); idx = {v: i for i, v in enumerate(nodes)}
    rr, cc = [], []
    for u, v in G.edges():
        rr += [idx[u], idx[v]]; cc += [idx[v], idx[u]]
    M = csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(len(nodes), len(nodes)))
    match = maximum_bipartite_matching(M, perm_type='column')
    return max(len(nodes) - int(np.sum(match >= 0)), 1)
agree = all(nd_scipy(G) == driver_nodes_matching(G) for kind in KINDS for G in GRAPHS[kind])
print('2. matching count identical under SciPy matching:', agree)

# 3. Why the ring lattice needs three under the exact count. Its eigenvalues come
#    in pairs, and at N = 100 the value -2 occurs three times, at j = 25, 50, 75.
nd, lam = driver_nodes_exact(GRAPHS['regular'][0])
print(f'3. ring lattice at N = 100: largest multiplicity {nd}, eigenvalue {lam:.4f}')
for n_alt in (98, 99, 101, 102):
    nd_alt, lam_alt = driver_nodes_exact(nx.watts_strogatz_graph(n_alt, K_MEAN, 0.0, seed=SEED0))
    print(f'   ring lattice at N = {n_alt}: exact count {nd_alt}')

# 4. Random realisations with elements outside the largest component.
out = col('random', 'elements_outside_largest_component')
on_giant_m = [driver_nodes_matching(giant(G)) for G in GRAPHS['random']]
on_giant_e = [driver_nodes_exact(giant(G))[0] for G in GRAPHS['random']]
print(f'4. random: {int(np.sum(out > 0))} realisations hold {int(out.sum())} elements outside '
      f'the largest component; on the largest component alone the counts average '
      f'{np.mean(on_giant_m):.2f} by matching and {np.mean(on_giant_e):.2f} exactly')

two = [r['seed'] for r in rows if r['topology'] == 'random' and r['driver_nodes_matching'] == 2]
print(f"   the random realisation needing two drivers, seed {two}, holds "
      f"{[r['elements_outside_largest_component'] for r in rows if r['topology'] == 'random' and r['seed'] in two]} "
      f"elements outside the largest component")

# 5. The spread of the scale-free counts, which a mean of about two hides.
for key, name in (('m', 'matching'), ('e', 'exact')):
    v = SUMMARY['scalefree'][key]
    print(f"5. scale-free, {name}: median {np.median(v):.0f}, one driver in {np.mean(v == 1) * 100:.0f} %, "
          f"four or more in {int(np.sum(v >= 4))} of 100, largest {v.max():.0f}")

# 6. C2 under other choices: degree shares, rank correlation, and the scaling constant.
print('6. mean C2 under other choices')
for kind in KINDS[1:]:
    print(f"   {LABEL[kind]:12s} eigenvector {col(kind, 'r_share_avgctrl').mean():.3f} | "
          f"degree shares {col(kind, 'r_degreeshare_avgctrl').mean():.3f} | "
          f"Spearman {col(kind, 'spearman_share_avgctrl').mean():.3f} | "
          f"scale 0.5 {col(kind, 'r_share_avgctrl_c0.5').mean():.3f} | "
          f"scale 2 {col(kind, 'r_share_avgctrl_c2').mean():.3f}")

# 7. Straight line, not strict proportion. Within a realisation average controllability
#    spans a narrow range and a fitted line keeps a large intercept.
print('7. average controllability against share, within realisations')
for kind in KINDS[1:]:
    fold, icpt, aa, bb, mm = [], [], [], [], []
    for G in (giant(Gf) for Gf in GRAPHS[kind]):
        p = coupling_shares(G, 'eigenvector'); ac = average_controllability(G)
        b, a = np.polyfit(p, ac, 1)
        fold.append(ac.max() / ac.min()); icpt.append(a / ac.mean())
        aa.append(a); bb.append(b); mm.append(ac.mean())
    print(f"   {LABEL[kind]:12s} largest over smallest {np.mean(fold):.2f}, "
          f"intercept {np.mean(icpt) * 100:.0f} % of the mean | mean line {np.mean(aa):.3f} + "
          f"{np.mean(bb):.1f} x share, mean average controllability {np.mean(mm):.3f}")

1. exact count unchanged between tolerances 1e-6 and 1e-9: True
2. matching count identical under SciPy matching: True
3. ring lattice at N = 100: largest multiplicity 3, eigenvalue -2.0000
   ring lattice at N = 98: exact count 6
   ring lattice at N = 99: exact count 2
   ring lattice at N = 101: exact count 2
   ring lattice at N = 102: exact count 3
4. random: 25 realisations hold 26 elements outside the largest component; on the largest component alone the counts average 1.00 by matching and 1.00 exactly
   the random realisation needing two drivers, seed [127], holds [2] elements outside the largest component
5. scale-free, matching: median 1, one driver in 60 %, four or more in 19 of 100, largest 10
5. scale-free, exact: median 1, one driver in 58 %, four or more in 19 of 100, largest 10
6. mean C2 under other choices
   Small-world  eigenvector 0.757 | degree shares 0.901 | Spearman 0.711 | scale 0.5 0.806 | scale 2 0.720
   Random       eigenvector 0.964 | degree shares 0.984 

## Numbers for the text

The values quoted in the manuscript and the Supplementary Information, printed once from the arrays above.

In [9]:
def fmt(v, d=2):
    return f'{v:.{d}f}'

print('C1  mean driver nodes, matching / exact')
for kind in KINDS:
    s = SUMMARY[kind]
    print(f"    {LABEL[kind]:16s} {fmt(s['m'].mean())} / {fmt(s['e'].mean())}   "
          f"range {s['m'].min():.0f}-{s['m'].max():.0f} / {s['e'].min():.0f}-{s['e'].max():.0f}")
lo = min(min(SUMMARY[k]['m'].mean(), SUMMARY[k]['e'].mean()) for k in KINDS)
hi = max(max(SUMMARY[k]['m'].mean(), SUMMARY[k]['e'].mean()) for k in KINDS)
print(f"    every arrangement, both counts: {lo:.2f} to {hi:.2f} of {N} on average")
print(f"    effective count 2^H: {effective_dof(H_EIG['regular'].mean()):.1f} regular against "
      f"{effective_dof(H_EIG['scalefree'].mean()):.1f} scale-free, a gap of "
      f"{effective_dof(H_EIG['regular'].mean()) - effective_dof(H_EIG['scalefree'].mean()):.1f}")
print('C2  mean Pearson r, share against average controllability')
for kind in KINDS:
    r = SUMMARY[kind]['r']
    print(f"    {LABEL[kind]:16s} " + ('undefined' if np.all(np.isnan(r)) else
          f"{r.mean():.2f}  (range {r.min():.2f} to {r.max():.2f})"))

C1  mean driver nodes, matching / exact
    Regular lattice  1.00 / 3.00   range 1-1 / 3-3
    Small-world      1.00 / 1.00   range 1-1 / 1-1
    Random           1.01 / 1.01   range 1-2 / 1-2
    Scale-free       2.17 / 2.24   range 1-10 / 1-10
    every arrangement, both counts: 1.00 to 3.00 of 100 on average
    effective count 2^H: 100.0 regular against 79.8 scale-free, a gap of 20.2
C2  mean Pearson r, share against average controllability
    Regular lattice  undefined
    Small-world      0.76  (range 0.55 to 0.88)
    Random           0.96  (range 0.94 to 0.98)
    Scale-free       0.97  (range 0.95 to 0.98)
